# Module 6.4: Capstone 3: Exporting & Quantizing Models to ONNX for Edge Hardware
### 🇵🇱 Projekt Zwieńczeniowy 3: Eksport i Kwantyzacja Modeli do ONNX na Urządzenia Brzegowe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m06_l04_exporting_quantizing_onnx_capstone3.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Convert deep vision models to open ONNX format, apply INT8 post-training quantization, and benchmark sub-5ms edge inference on industrial IPCs.  
> 🇵🇱 **Streszczenie:** Konwertuj modele do formatu ONNX, zastosuj kwantyzację INT8 i osiągnij opóźnienie poniżej 5ms na komputerach przemysłowych IPC.

- **Estimated Time:** 30 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Capstone 3: Edge Model Optimization & ONNX Quantization
1. **Industrial Edge Hardware**: Fanless industrial PCs (Siemens IPC) and edge micro-servers (NVIDIA Jetson) have constrained RAM and 15–25W thermal budgets.
2. **Open Neural Network Exchange (ONNX)**: Decouples model graphs from Python runtimes, enabling direct C++ execution.
3. **INT8 Post-Training Quantization (PTQ)**: Converts 32-bit floating point weights into 8-bit signed integers ($[-128, 127]$). Reduces model footprint by $75\%$ and accelerates inference latency by $3	imes-4	imes$.


In [ ]:
# 1. Simulate Edge Deep Learning Model Weights & INT8 Quantization
np.random.seed(42)
n_weights = 250_000

# FP32 weights from trained convolutional inspection model
fp32_weights = np.random.normal(loc=0.0, scale=0.45, size=n_weights).astype(np.float32)

# Compute symmetric INT8 quantization scale factor
max_val = np.max(np.abs(fp32_weights))
scale = max_val / 127.0

# Quantize to INT8: q = round(w / scale)
int8_weights = np.clip(np.round(fp32_weights / scale), -128, 127).astype(np.int8)

# Dequantize for reconstruction accuracy check: w_approx = q * scale
dequant_weights = int8_weights.astype(np.float32) * scale

# Calculate memory sizes
size_fp32_kb = fp32_weights.nbytes / 1024
size_int8_kb = int8_weights.nbytes / 1024

print(f"FP32 Model Memory : {size_fp32_kb:.1f} KB")
print(f"INT8 Model Memory : {size_int8_kb:.1f} KB ({(1.0 - size_int8_kb/size_fp32_kb)*100:.1f}% reduction)")


In [ ]:
# 2. Benchmark Simulated Inference Latency on Industrial Edge IPC
# Simulate 500 edge inference runs
latency_fp32_ms = np.random.normal(14.8, 1.2, 500)   # FP32 CPU execution: ~15ms
latency_int8_ms = np.random.normal(3.9, 0.4, 500)    # INT8 quantized C++ execution: ~4ms

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory footprint comparison
axes[0].bar(["FP32 Original", "INT8 Quantized"], [size_fp32_kb, size_int8_kb], color=['#ef4444', '#10b981'], width=0.45)
axes[0].set_title("Model Memory Footprint Reduction", fontweight='bold')
axes[0].set_ylabel("RAM Allocation [KB]")
axes[0].grid(axis='y', alpha=0.3)

# Latency benchmark distribution
sns.kdeplot(latency_fp32_ms, color='#ef4444', fill=True, alpha=0.3, label=f'FP32 Baseline (Mean = {np.mean(latency_fp32_ms):.1f}ms)', ax=axes[1])
sns.kdeplot(latency_int8_ms, color='#10b981', fill=True, alpha=0.3, label=f'ONNX INT8 Edge (Mean = {np.mean(latency_int8_ms):.1f}ms)', ax=axes[1])
axes[1].axvline(5.0, color='#f59e0b', linestyle='--', label='Conveyor Real-Time Cycle Limit (5ms)')
axes[1].set_title("Inference Latency Benchmark on Siemens IPC", fontweight='bold')
axes[1].set_xlabel("Latency [milliseconds]")
axes[1].set_ylabel("Probability Density")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Export trained PyTorch model to Open Neural Network Exchange (ONNX) graph *(🇵🇱 Wyeksportuj wytrenowany model PyTorch do otwartego formatu ONNX)*
- [ ] **Task 2:** Apply INT8 Post-Training Quantization (PTQ) to compress weights and activations *(🇵🇱 Zastosuj kwantyzację INT8 po treningu (PTQ), redukując wagi i aktywacje)*
- [ ] **Task 3:** Benchmark latency and throughput (FPS) in ONNX Runtime across FP32 and INT8 *(🇵🇱 Zmierz opóźnienie i przepustowość (FPS) w ONNX Runtime dla wersji FP32 i INT8)*


In [ ]:
# Exercise Challenge:
# Assert that INT8 quantization slashes mean latency by at least 3x and stays under 5ms
speedup = np.mean(latency_fp32_ms) / np.mean(latency_int8_ms)
mean_int8 = np.mean(latency_int8_ms)

print(f"Measured Speedup Factor : {speedup:.2f}x")
print(f"Mean INT8 Latency       : {mean_int8:.2f} ms")

assert speedup >= 3.0, "INT8 quantization failed to deliver 3x speedup!"
assert mean_int8 < 5.0, "Edge latency exceeds 5ms conveyor real-time limit!"
print("✓ Capstone 3 ONNX edge quantization verified.")
